# 14 · One score per consignment

**The question.** DESIGN_QUESTIONS S6.1: should the review queue rank line items or whole consignments? Under rule D, 10.9% of the top-1% slots go to a record whose permit already has a record in the top 1%, and 85% of the multi-record permits there are split by the cut (notebook 05, M9). Ranking consignments needs a rule that turns the scores of a consignment's records into one score, and report §4.8 says that rule cannot be tested on LEMIS. It can: a LEMIS shipment groups line items the way a CITES permit groups records (notebook 09), and the pipeline already scores shipments by their highest line.

**Fixed before looking:**
1. **The LEMIS stand-in for a permit:** the listed line items of a 2011 shipment (a CITES permit holds only listed specimens). Its label: whether any of them is an enforcement case.
2. **Line scores:** the chosen ensemble (XGBoost and a 100-tree random forest, rank-averaged, trained on 2000–2010 with the 21 shared features; notebooks 10–12).
3. **Three ways to combine them:**
   - the **maximum** line score;
   - the **mean of the top two** line scores (a single-line shipment keeps its score);
   - **noisy-or**, 1 − ∏(1 − p), on each model's line probabilities, then the two models' shipment scores rank-averaged. It rises with the number of high lines.
4. **Rule:** the way with the highest recall at a 1% review budget of shipments is chosen; one whose 95% bootstrap interval (shipments resampled) against the best includes zero counts as tied, and ties go to the maximum, the simplest. Recall at 5% is reported alongside.
5. **The unit itself** (line items or consignments) is then decided from this result and M9; it changes what the reviewer opens, not the model.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

import eda_setup as E
import experiment_setup as X_
import model_selection as M
from wildguard import config
from wildguard.features import SHARED_FEATURES as FEATS, FeatureBuilder

E.style()
t0 = time.time()
df = X_.load_lemis()
df = df[df["animal"]].reset_index(drop=True)
tr, ev = df["year"].between(2000, 2010).to_numpy(), (df["year"] == 2011).to_numpy()
search = pd.read_parquet(config.PROCESSED_DIR / "experiments" / "model_search.parquet")
row = search[(search.candidate == "xgboost") & (search.config == 5)].iloc[0]
fb = FeatureBuilder(use_value=False).fit(df[tr])
F_tr, F_ev = fb.transform(df[tr]), fb.transform(df[ev])
base = M.Base().fit(F_tr, FEATS)
Xtr, Xev = base.transform(F_tr), base.transform(F_ev)
enc = M.Native(base).fit(Xtr)
y = df["label"].to_numpy()[tr]
xgb = M.build_model("xgboost", M.config_from_row(row.to_dict()), enc, y, 37, int(row["best_rounds"]))
xgb.fit(enc.transform(Xtr), y)
rf = RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=5, max_features="sqrt",
                            n_jobs=8, random_state=37).fit(Xtr, y)
lines = df[ev][["shipment_id", "label", "sp_listed"]].reset_index(drop=True)
lines["p_xgb"] = xgb.predict_proba(enc.transform(Xev))[:, 1]
lines["p_rf"] = rf.predict_proba(Xev)[:, 1]
lines = lines[lines["sp_listed"]].reset_index(drop=True)
lines["score"] = (lines.p_xgb.rank(pct=True) + lines.p_rf.rank(pct=True)) / 2
print(f"{len(lines):,} listed 2011 line items in {lines.shipment_id.nunique():,} shipments "
      f"({time.time() - t0:.0f}s)")

101,932 listed 2011 line items in 41,076 shipments (105s)


## 1. Shipment scores three ways

In [2]:
g = lines.groupby("shipment_id")
ship = pd.DataFrame({"label": g["label"].max(), "lines": g.size(),
                     "maximum": g["score"].max(),
                     "mean of top two": g["score"].apply(lambda s: s.nlargest(2).mean())})
nor = pd.DataFrame({m: g[m].apply(lambda p: 1 - np.prod(1 - p)) for m in ("p_xgb", "p_rf")})
ship["noisy-or"] = (nor.p_xgb.rank(pct=True) + nor.p_rf.rank(pct=True)) / 2
ship = ship.reset_index()
print(f"{len(ship):,} shipments, {int(ship.label.sum()):,} with an enforcement case; "
      f"{(ship.lines > 1).mean():.1%} have more than one listed line item")
WAYS = ["maximum", "mean of top two", "noisy-or"]


def recall_at(yv, s, frac, w=None):
    w = np.ones(len(s)) if w is None else w
    order = np.argsort(-s, kind="stable")
    top = np.cumsum(w[order]) <= frac * w.sum()
    return (w[order][top] * yv[order][top]).sum() / (w * yv).sum()


yv = ship["label"].to_numpy()
rng = np.random.default_rng(37)
boot = {k: [] for k in WAYS}
for _ in range(1000):
    w = np.bincount(rng.integers(0, len(ship), len(ship)), minlength=len(ship)).astype(float)
    for k in WAYS:
        boot[k].append(recall_at(yv, ship[k].to_numpy(), 0.01, w))
res = pd.DataFrame({k: {"recall at 1%": recall_at(yv, ship[k].to_numpy(), 0.01),
                        "recall at 5%": recall_at(yv, ship[k].to_numpy(), 0.05)} for k in WAYS}).T
best = res["recall at 1%"].idxmax()
for k in WAYS:
    d = np.array(boot[k]) - np.array(boot[best])
    res.loc[k, f"− {best}"], res.loc[k, "95% low"], res.loc[k, "95% high"] = d.mean(), *np.percentile(d, [2.5, 97.5])
res["tied with the best"] = res["95% high"] >= 0
display(res.style.format({c: "{:.3f}" for c in res.columns[:5]}))
CHOICE = "maximum" if res.loc["maximum", "tied with the best"] else best
print(f"Best at 1%: {best}; choice by the tie rule: {CHOICE}")

41,076 shipments, 1,795 with an enforcement case; 32.9% have more than one listed line item


,recall at 1%,recall at 5%,− maximum,95% low,95% high,tied with the best
maximum,0.228,0.617,0.000,0.000,0.000,True
mean of top two,0.228,0.607,-0.000,-0.002,0.001,True
noisy-or,0.214,0.597,-0.013,-0.018,-0.008,False


Best at 1%: maximum; choice by the tie rule: maximum


## 2. What ranking consignments changes for a reviewer

The same 1% budget, spent on line items or on shipments: how many shipments the reviewer opens, and what share of the shipments with a case they reach.

In [3]:
k_lines = int(round(0.01 * len(lines)))
top_lines = lines.nlargest(k_lines, "score")
k_ship = int(round(0.01 * len(ship)))
top_ship = ship.nlargest(k_ship, CHOICE)
pd.DataFrame({
    "line items": {"review slots": k_lines, "distinct shipments opened": top_lines.shipment_id.nunique(),
                   "shipments with a case reached": ship.set_index("shipment_id").loc[top_lines.shipment_id.unique(), "label"].sum()},
    f"shipments ({CHOICE})": {"review slots": k_ship, "distinct shipments opened": k_ship,
                              "shipments with a case reached": top_ship.label.sum()}}).T

,review slots,distinct shipments opened,shipments with a case reached
line items,1019,876,794
shipments (maximum),411,411,410


## Findings

| # | Finding |
|---|---|
| W1 | 2011 has 101,932 listed line items in 41,076 shipments, 1,795 of them with an enforcement case; a third of the shipments (32.9%) have more than one listed line item |
| W2 | **Combining line scores.** At a 1% budget of shipments, the maximum line score reaches 22.8% of the shipments with a case, the mean of the top two the same (tied), noisy-or 21.4% (significantly worse, −0.013). At 5%: 61.7%, 60.7%, 59.7%. The rule chooses the **maximum** |
| W3 | **For a reviewer.** A 1% budget of line items is 1,019 review slots; they open 876 distinct shipments and reach 794 shipments with a case. A 1% budget of shipments is 411 slots and reaches 410. Ranking consignments spends no slot on a second line of a consignment already in the queue, and nearly every slot reaches a case |
| W4 | The rule for combining scores **can** be tested on LEMIS, through shipments; report §4.8 needs correcting |

**Decision on combining (S6.1):** a consignment's score is its highest line score. Whether the queue ranks consignments or line items is recorded in DESIGN_QUESTIONS.md.